# Notebook 4 — Feature Engineering

## Learning objectives

- Tell the five feature *types* apart (numeric, categorical, boolean, ordinal, cyclic).
- Apply standardisation, min-max and robust scaling, and choose between them.
- One-hot encode categorical features safely with `handle_unknown='ignore'`.
- Encode cyclic features with $(\sin, \cos)$ pairs and explain why this is the right geometry.
- Build lag features with `pandas.shift(+k)` and avoid lag-zero target leakage.
- Apply a chronological split, recognise where it is mandatory, and run a feature-set ablation.

## 4.1 Why engineer features at all?

A learning algorithm only learns patterns *expressible* in its model family from features that *exist* in its input. Linear regression cannot see a non-linear relationship in raw data unless we construct a feature that linearises it. Tree-based models cannot exploit the *circularity* of "hour of day" unless we encode that circularity. **Feature engineering** is the deliberate construction of inputs that reveal the structure already present in the data.


## 4.2 What is a feature?

We restate the definition from Notebook 1: a **feature** is a single column of the input matrix $\mathbf{X}$, a measurable property of one observation. Features can be:

- **Numeric** (continuous or integer): `TEMP`, `PRES`, `PM2.5`, `hour`.
- **Categorical** (a finite set of labels): `wd` (16 wind directions).
- **Boolean** (a special case of categorical with two levels): `is_rain` (0/1).
- **Ordinal** (categorical with order): `target_category_3` ∈ {Low, Medium, High}.
- **Cyclic** (numeric but topologically a circle): `hour` ∈ {0,…,23} where 23 and 0 are *adjacent*.

Each type demands a different encoding before it can be used by a typical numeric model. We load the forecasting-ready dataset to see them in action.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path('data')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

csv = DATA_DIR / 'air_quality_nongzhanguan_forecast.csv'
print('Loading:', csv.name)
df = pd.read_csv(csv, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
print('Rows:', len(df), '| Columns:', len(df.columns))
df.head(3)

## 4.3 Numeric feature scaling

Three standard families:

**Standardisation** — the default.

$$\tilde{x}_{ij} = \frac{x_{ij} - \mu_{j}}{\sigma_{j}}.$$

After standardisation every feature has mean 0 and standard deviation 1.

**Min-max scaling** — useful when downstream layers expect bounded inputs.

$$\tilde{x}_{ij} = \frac{x_{ij} - \min_{j}}{\max_{j} - \min_{j}}.$$

Sensitive to outliers because $\min$ and $\max$ are.

**Robust scaling** — for skewed atmospheric variables (e.g., RAIN, which is zero $> 95\%$ of the time).

$$\tilde{x}_{ij} = \frac{x_{ij} - \text{median}_{j}}{\text{IQR}_{j}}.$$

Which to choose? *Default to standardisation*. Use min-max when downstream layers (in neural netowk models) expect bounded inputs. Use robust scaling when features have heavy tails. The project uses `StandardScaler` throughout.

> **The scaler is fitted on training data only.** Refitting per-fold inside cross-validation is a strict requirement, automated by `sklearn.Pipeline`. Forgetting this is the single most common source of *information leakage* in beginner ML pipelines.

## 4.4 Categorical feature encoding

Categorical encoding is the essential transformation of qualitative data (such as categories, labels, or strings) into numerical formats that machine learning algorithms can process mathematically. Selecting the appropriate approach depends on the nature of your categories and the requirements of your chosen model. The four standard encodings are:

### 4.4.1 One-hot encoding (default)

A feature with $K$ levels becomes $K$ binary indicator columns, exactly one of which is 1 for any given row. Wind direction `wd` ($K = 16$) becomes 16 columns `wd_N`, `wd_NNE`, ….

Pros: no spurious ordering imposed; works with every model family.

Cons: blows up dimensionality for high-cardinality features; introduces collinearity (the columns sum to 1).

### 4.4.2 Ordinal (label) encoding

Map each category to a single integer. Suitable only when the categories have a true order *and* the distance between consecutive levels is meaningful. For nominal categories like wind direction, ordinal encoding would impose a phantom order and bias linear models.

### 4.4.3 Frequency encoding

Replace each category by its empirical frequency in the training set. Lossy but compact, surprisingly effective for tree-based models on high-cardinality features.

### 4.4.4 Target / mean encoding

Replace each category by the mean of $y$ within that category. Powerful but leak-prone: must be computed from the training fold only.

The project applies one-hot via `sklearn.preprocessing.OneHotEncoder`, embedded in a `ColumnTransformer`. The `handle_unknown='ignore'` argument is essential for time-series deployment: if an unseen wind-direction string appears in production, the encoder emits a row of zeros rather than throwing an exception.

## 4.5 Cyclic encoding

Some numeric features are *topologically circular*. Hour of day runs 0, 1, ..., 23, then wraps back to 0. Hour 23 is closer to hour 0 than hour 23 is to hour 12 - a fact a model cannot infer from the integer representation alone, because as numbers $|23 - 0| = 23 \gg |23 - 12| = 11$.

The fix is to map the cyclic feature to a pair of $\sin / \cos$ coordinates on the unit circle:

$$x_{\sin} = \sin\left(\frac{2\pi t}{P}\right), \qquad x_{\cos} = \cos\left(\frac{2\pi t}{P}\right),$$

where $P$ is the period. For hour-of-day $P = 24$; for month-of-year $P = 12$; for day-of-week $P = 7$.

The pair $(x_{\sin}, x_{\cos})$ has two crucial properties:

1. The Euclidean distance between the encodings of hour 23 and hour 0 equals the distance between hour 11 and hour 12 — both equal $2\sin(\pi/24)$.
2. Any function of $t$ that is smooth and periodic with period $P$ can be approximated by a linear combination of $\sin(2\pi t / P)$ and $\cos(2\pi t / P)$ (a first-order Fourier expansion).

The project applies this to all three temporal variables. We can plot the resulting points on the unit circle to visualise the encoding.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(df['hour_sin'], df['hour_cos'], c=df['hour'], cmap='twilight', s=8)
axes[0].set_title('hour-of-day on the unit circle'); axes[0].set_xlabel('sin'); axes[0].set_ylabel('cos'); axes[0].set_aspect('equal')
axes[1].scatter(df['month_sin'], df['month_cos'], c=df['month'], cmap='twilight', s=8)
axes[1].set_title('month-of-year'); axes[1].set_xlabel('sin'); axes[1].set_ylabel('cos'); axes[1].set_aspect('equal')
axes[2].scatter(df['day_of_week_sin'], df['day_of_week_cos'], c=df['day_of_week'], cmap='twilight', s=8)
axes[2].set_title('day-of-week'); axes[2].set_xlabel('sin'); axes[2].set_ylabel('cos'); axes[2].set_aspect('equal')
plt.tight_layout(); plt.show()

Each colour ring you see is the $K$ distinct values of the underlying integer feature wrapped onto a circle. Midnight (hour 0) and 23:00 sit next to each other in the encoding even though they are 23 units apart in raw integer form.

## 4.6 Lag features

Atmospheric variables are highly autocorrelated: $\text{PM}_{2.5}$ at hour $t$ is strongly predictable from $\text{PM}_{2.5}$ at hour $t-1$. A **lag feature** is the value of a variable at an earlier time, attached as a column to the row at time $t$:

$$x^{\text{lag-}k}_{t} = x_{t-k}.$$

For pollutant forecasting we use multiple lags simultaneously: $\{\text{lag-}1, \text{lag-}2, \text{lag-}3, \text{lag-}6, \text{lag-}12, \text{lag-}24\}$ for the target pollutant; $\{\text{lag-}1, \text{lag-}6, \text{lag-}24\}$ for the others. The "rich" target lags expose short-term momentum (lag-1, lag-2), medium-term diurnal echo (lag-6, lag-12), and the previous-day-same-hour cycle (lag-24).

The project implements lag construction with `pandas.DataFrame.shift`:

```python
for lag in target_lags:
    df[f'{target}_lag_{lag}'] = df[target].shift(lag)
```

`shift(k)` with positive $k$ moves values *down* the column (so row $t$ now contains the value originally at $t-k$). The first $k$ rows therefore become `NaN` and are dropped before training.

> **Critical rule.** Lag features must be `shift(+k)` with $k \ge 1$, never $k = 0$ (that would equal the current value, leaking the target if the target *is* the lagged variable).

In [ ]:
lag_cols = [c for c in df.columns if '_lag_' in c]
target_lags = [c for c in lag_cols if c.startswith('PM2.5_lag_')]
print('Total lag columns:', len(lag_cols))
print('PM2.5 target lags :', target_lags)

In [ ]:
# Quick sanity check: PM2.5_lag_1 row t should equal PM2.5 row t-1.
sample = df[['datetime', 'PM2.5', 'PM2.5_lag_1', 'PM2.5_lag_24']].head(30)
sample

## 4.7 Time-based data division

With autocorrelated features, the split must be chronological:

$$\underbrace{\mathcal{D}_{\text{train}}}_{t \in [t_{0}, t_{\text{cut}}]} \quad\text{and}\quad \underbrace{\mathcal{D}_{\text{test}}}_{t \in (t_{\text{cut}}, t_{\text{end}}]}.$$

For the 33,070-row $\text{PM}_{2.5}$ forecasting dataset at *Nongzhanguan*, `test_size=0.2` puts 26,456 rows in train (2013-03-08 → 2016-05-29) and 6,614 rows in test (2016-05-29 → 2017-02-28). The test period contains an entire winter, capturing the worst pollution episodes — this is realistic, because that is exactly when forecasting matters most.

For *cross-validation* on time-series data, the standard tool is `TimeSeriesSplit`, which produces forward-only folds (we use it in Notebook 6). `KFold` and `train_test_split(shuffle=True)` are *forbidden* for forecasting models.

In [ ]:
cut = int(0.8 * len(df))
train_df, test_df = df.iloc[:cut], df.iloc[cut:]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train_df['datetime'], train_df['PM2.5'], label='Train', lw=0.8)
ax.plot(test_df['datetime'],  test_df['PM2.5'],  label='Test',  lw=0.8, color='tab:red')
ax.set_ylabel('PM2.5 (μg/m³)'); ax.set_title('Chronological 80/20 split')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Train: {train_df.datetime.min()} -> {train_df.datetime.max()}')
print(f'Test : {test_df.datetime.min()}  -> {test_df.datetime.max()}')

## 4.8 Real-world application — feature-set ablation

Eight feature sets are commonly compared on this kind of data. The three most informative for our purposes are:

| Feature set | Composition |
|---|---|
| `baseline` | current-hour numeric + cyclic + wind direction |
| `lags_only` | all lag features + cyclic |
| `baseline_plus_lags` | union of the two |

We train one **linear regression** per feature set, all targeting `target_pm25_h1` (one-hour-ahead $\text{PM}_{2.5}$). The numerical contrast is this notebook's empirical anchor.

In [ ]:
TARGET = 'target_pm25_h1'
CYCLIC = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']
METEO  = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
CURRENT_NUMERIC = ['PM10', 'SO2', 'NO2', 'CO', 'O3'] + METEO

feature_sets = {
    'baseline':            CURRENT_NUMERIC + CYCLIC,
    'lags_only':           lag_cols + CYCLIC,
    'baseline_plus_lags':  CURRENT_NUMERIC + CYCLIC + lag_cols,
}
for k, v in feature_sets.items():
    print(f'{k:>22s}  {len(v)} numeric features')

In [ ]:
def evaluate(numeric_features, df_train, df_test, target=TARGET):
    pre = ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
    ])
    pipe = Pipeline([('pre', pre), ('lr', LinearRegression())])
    pipe.fit(df_train[numeric_features + ['wd']], df_train[target])
    pred = pipe.predict(df_test[numeric_features + ['wd']])
    return mean_absolute_error(df_test[target], pred), r2_score(df_test[target], pred)

results = []
for name, feats in feature_sets.items():
    mae, r2 = evaluate(feats, train_df, test_df)
    results.append({'feature_set': name, 'n_features': len(feats), 'MAE': mae, 'R2': r2})
results_df = pd.DataFrame(results).set_index('feature_set')
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results_df['MAE'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('MAE (lower is better)'); axes[0].set_ylabel('μg/m³')
results_df['R2'].plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('R² (higher is better)')
for ax in axes:
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

**Interpretation.** `baseline_plus_lags` wins; lags alone are nearly as good; current-hour features alone trail by several MAE points. When this comparison is rerun across multiple pollutants and horizons, **`baseline_plus_lags` with a gradient-boosting model typically wins** — we will reproduce a slice of this experiment in Notebook 5. Lags carry most of the signal at $h=1$; meteorology starts to matter more at longer horizons (Notebook 5).

## 4.9 Common misconceptions

- **"Cyclic encoding is just two features instead of one."** It is, but the *information geometry* changes: distances in $(\sin, \cos)$ space are circular distances in the original variable. Without the encoding, a tree-based model would split "hour < 12" and lose the equivalence of midnight and 23:59.
- **"Lag features make a model 'use the past'."** They do, but only because *we* hand-fed them. The model itself has no temporal awareness; it sees a row of numbers and predicts. The temporal structure is entirely in the feature matrix.
- **"Standardisation should be done on the whole dataset for stability."** No. Always fit on training data only. Refitting per-fold inside cross-validation is a strict requirement, automated by `sklearn.Pipeline`.
- **"You can lag the target by zero hours."** That would be the target itself. Set $k \ge 1$ always; for $h$-step-ahead forecasting, the *target* is `target.shift(-h)` (negative shift ⇒ future), but the *lag features* are still `shift(+k)` for $k \ge 1$.

---
## Exercises

### Exercise 4.1 — visualise a single hour-of-day's diurnal cycle

*Hint: `df.groupby('hour')['PM2.5'].mean()` plots a 24-hour mean. Why is encoding via $(\sin, \cos)$ the right way to feed this curve to a linear regression?*

In [ ]:
# EXERCISE 4.1
# diurnal = df.groupby('hour')['PM2.5'].mean()
# TODO: plot diurnal; comment on why integer hour 0 vs 23 looks distant numerically but adjacent physically.


### Exercise 4.2 — try `target_lags_meteo`

*Hint: build a feature set of just `target_lags + METEO + CYCLIC + ['wd']`. Does it beat `lags_only`? It should — adding meteo never hurts at $h=1$.*

In [ ]:
# EXERCISE 4.2
# feats = target_lags + METEO + CYCLIC
# TODO: call evaluate(feats, train_df, test_df) and compare MAE/R² to the three default sets.


### Exercise 4.3 — leakage stress test

*Hint: replace `target.shift(-1)` with `target.shift(0)` in your own implementation. Why does $R^2$ jump to 1.0? Why is this catastrophic?*

In [ ]:
# EXERCISE 4.3
# y_leak = train_df['PM2.5']  # zero-step shift = current target
# TODO: train regression with y_leak as target; observe R² and explain the failure mode.


## 4.10 Chapter summary

- Feature engineering is the deliberate transformation of raw inputs into forms a model can exploit; the choice of transformation is at least as important as the choice of model family.
- Numeric features are standardised (or robust-scaled, or min-max-scaled), categorical features are one-hot encoded, ordinal features are integer-encoded, and *cyclic* features are encoded as $(\sin, \cos)$ pairs.
- Lag features turn an autocorrelated time series into a tabular regression problem; we use lags $\{1, 2, 3, 6, 12, 24\}$ on the target pollutant and $\{1, 6, 24\}$ on the others.
- Time-based data division is the only honest evaluation protocol for forecasting; random splits leak future information into training.
- Every transformation must be fitted on training data only, and re-applied at test time with the *same* parameters; `sklearn.Pipeline` makes this automatic.

**Next:** Notebook 5 puts these features to work in *direct forecasting* at multiple horizons.